In [1]:
import os 
os.chdir("../")
%pwd

'd:\\Programming\\ML\\End-to-End\\End-to-End-TelcoChurn'

In [2]:
from dataclasses import dataclass 
from pathlib import Path 

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir:Path 
    model_name: str
    target_column: str 
    all_params:dict
    X_train_path : Path 
    X_test_path : Path
    y_train_path : Path
    y_test_path :Path
    THRESHOLD : float
    best_params_path : Path
    
    

In [3]:
from src.constants import *
from src.utils import read_yaml, create_directories

class ConfigurationManager:
    def __init__(self,
                config_path = Path(CONFIG_FILE_PATH),
                params_path = Path(PARAMS_FILE_PATH),
                schema_path = Path(SCHEMA_FILE_PATH)):
        self.config = read_yaml(config_path)
        self.params = read_yaml(params_path) 
        self.schema = read_yaml(schema_path)
        
        create_directories([self.config.artifacts_root])
        

    def get_model_trainer(self) -> ModelTrainerConfig:
        config = self.config.model_trainer 
        params = self.params.xgboost_params
        schema = self.schema.TARGET_COLUMN
        
        create_directories([config.root_dir])
        
        return ModelTrainerConfig(
            root_dir= Path(config.root_dir),
            model_name=config.model_name,
            target_column = schema.name,
            all_params = params,
            X_train_path = Path(config.X_train_path),
            X_test_path = Path(config.X_test_path),
            y_train_path= Path(config.y_train_path),
            y_test_path= Path(config.y_test_path),
            best_params_path= Path(config.best_params_path),
            THRESHOLD = config.THRESHOLD,
        )

In [ ]:
import pandas as pd
import os
from src import logging, CustomException
from xgboost import XGBClassifier
import joblib
import time
from scipy import sparse
from sklearn.metrics import recall_score
import optuna
import yaml
from dataclasses import replace


class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config 
        
        
    def get_best_params(self, n_trials=30):
        """Optuna hyperparameter tuning"""
        logging.info("Find best params")
        params_config = self.config.all_params
        
        # load data 
        X_train_arr = sparse.load_npz(self.config.X_train_path)
        X_test_arr = sparse.load_npz(self.config.X_test_path)
        y_train = pd.read_csv(self.config.y_train_path)
        y_test = pd.read_csv(self.config.y_test_path)
        
        def objective(trial):
            params = {} 
            for key, cfg in params_config.items():
                if isinstance(cfg, (int, float, str, bool)):
                    params[key] = cfg 
                    continue
                
                p_type = cfg.get("type")
                low = cfg.get("low")
                high = cfg.get("high")
                
                if p_type == "int":
                    params[key] = trial.suggest_int(key,low,high)
                    
                elif p_type == "float":
                    if cfg.get("log", False):
                        params[key] = trial.suggest_float(key, low, high, log=True)
                    else:
                        params[key] = trial.suggest_float(key, low, high)
                
            # handle dynamic scale_pos_weight 
            if params.get("scale_pos_weight") == "auto":
                scale = (y_train.value_counts().iloc[0]).sum() / (y_train.value_counts().iloc[1]).sum()
                params["scale_pos_weight"] = float(scale)
                
            # Fixed values 
            params["random_state"] = 42 
            params["n_jobs"] = -1 
            params["eval_metric"] = "logloss"
            
            
            # Evaluate model 
            model = XGBClassifier(**params)       
            model.fit(X_train_arr, y_train)
            proba = model.predict_proba(X_test_arr)[:,1]
            y_pred = (proba >= self.config.THRESHOLD).astype(int)
            return recall_score(y_test ,y_pred, pos_label=1)
        
        
        # run optuna 
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=n_trials)     
        
        
        best_params = study.best_params 
        logging.info(f"✅ Best parameters found: {best_params}")
        
        # save best params 
        with open(self.config.best_params_path, "w") as f:
            yaml.dump(best_params, f)
        
        # update config to use best params for training
        self.config= replace(self.config, all_params=best_params)
        
        return best_params
            
    
    
    def train(self):
        """Train final model using best params"""
        #Initiate Model 
        model = XGBClassifier(**self.config.all_params)
        logging.info(f"🚀 Initializing model: {model.__class__.__name__} with parameters: {model.get_params()}")
        
        # load data 
        X_train_arr = sparse.load_npz(self.config.X_train_path)
        y_train = pd.read_csv(self.config.y_train_path)
        
        # Training timer
        start_train = time.time()
        model.fit(X_train_arr, y_train)
        train_time = time.time() - start_train
        print(f"⏱ Training time: {train_time:.2f} seconds")
        
        # save model 
        model_save_path = os.path.join(self.config.root_dir, self.config.model_name)
        joblib.dump(model, model_save_path)
        logging.info(f"Model {model.__class__.__name__} saved at {model_save_path}")
        
        return model
        
        

c:\Users\bedee\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
config = ConfigurationManager() 
model_trainer_config=config.get_model_trainer()
trainer = ModelTrainer(model_trainer_config) 
best_params = trainer.get_best_params() 
model = trainer.train()

[2025-10-16 09:30:39,178] [INFO] [root:read_yaml:16] - reading the content of 'config\config.yaml'
[2025-10-16 09:30:39,184] [INFO] [root:read_yaml:16] - reading the content of 'params.yaml'
[2025-10-16 09:30:39,189] [INFO] [root:read_yaml:16] - reading the content of 'schema.yaml'
[2025-10-16 09:30:39,191] [INFO] [root:create_directories:39] - created directory at: artifacts
[2025-10-16 09:30:39,193] [INFO] [root:create_directories:39] - created directory at: artifacts/model_trainer
[2025-10-16 09:30:39,194] [INFO] [root:get_best_params:21] - Find best params


[I 2025-10-16 09:30:39,219] A new study created in memory with name: no-name-f7ad2de6-447b-4091-a677-b28a79977e1a
[I 2025-10-16 09:30:40,537] Trial 0 finished with value: 0.9061662198391421 and parameters: {'n_estimators': 619, 'learning_rate': 0.18136774826495358, 'max_depth': 10, 'subsample': 0.7539723471140334, 'colsample_bytree': 0.5882922589658245, 'min_child_weight': 8, 'gamma': 2.792559991968276, 'reg_alpha': 3.268147542712496, 'reg_lambda': 0.7764202729280806}. Best is trial 0 with value: 0.9061662198391421.
[I 2025-10-16 09:30:41,131] Trial 1 finished with value: 0.9276139410187667 and parameters: {'n_estimators': 406, 'learning_rate': 0.18383011496675733, 'max_depth': 10, 'subsample': 0.9549958427650922, 'colsample_bytree': 0.5673274695707928, 'min_child_weight': 5, 'gamma': 3.9443157030565743, 'reg_alpha': 1.952452908919895, 'reg_lambda': 4.849268801311049}. Best is trial 1 with value: 0.9276139410187667.
[I 2025-10-16 09:30:42,896] Trial 2 finished with value: 0.88203753351

[2025-10-16 09:31:22,400] [INFO] [root:get_best_params:75] - ✅ Best parameters found: {'n_estimators': 416, 'learning_rate': 0.16545421590491707, 'max_depth': 4, 'subsample': 0.9958671027242365, 'colsample_bytree': 0.8340982748214413, 'min_child_weight': 9, 'gamma': 4.990452754436231, 'reg_alpha': 1.7480259244312855, 'reg_lambda': 1.6496239453791512}
[2025-10-16 09:31:22,407] [INFO] [root:train:92] - 🚀 Initializing model: XGBClassifier with parameters: {'objective': 'binary:logistic', 'base_score': None, 'booster': None, 'callbacks': None, 'colsample_bylevel': None, 'colsample_bynode': None, 'colsample_bytree': 0.8340982748214413, 'device': None, 'early_stopping_rounds': None, 'enable_categorical': False, 'eval_metric': None, 'feature_types': None, 'feature_weights': None, 'gamma': 4.990452754436231, 'grow_policy': None, 'importance_type': None, 'interaction_constraints': None, 'learning_rate': 0.16545421590491707, 'max_bin': None, 'max_cat_threshold': None, 'max_cat_to_onehot': None, 